# 03 — Excel and GeoPackage Export (Phase 3)

Converts the enriched global CSVs from Phase 2 into delivery formats suitable for country offices (tabular and GIS vector). Produces both per-country exports (filtered by ISO3 / ucode) and global exports, in Excel (.xlsx) and GeoPackage (.gpkg) formats.

## Inputs

- Global CSVs produced by notebook 02 (`{output_folder}/global/CSV/{hazard}_{date}.csv`)
- Admin2 GEE FeatureCollection for geometry attachment (used by the GeoPackage path)

## Outputs

Per-country: `{output_folder}/results/{country_ucode}/excel/{country}_{hazard}_{date}.xlsx` and `{output_folder}/results/{country_ucode}/geopackage/{country}_{hazard}_{date}.gpkg`. Global: `{output_folder}/global/excel/{hazard}_{date}.xlsx` and `{output_folder}/global/geopackage/{hazard}_{date}.gpkg`.

## Execution order

Run **after** notebook 02. Independent of notebooks 04 (rasters) and 05 (SDMX).


In [ ]:
# ============================================================
# CCRI Hazard Statistics Processing in GEE.
# Phase 3: Export in Excel and GeoJSON formats. 
# Author: Angelly Pugliese, Ph.D.
# Date: February 2026
# ============================================================

In [ ]:
# ============================================================
# Imports
# ============================================================
import pandas as pd
import geopandas as geopd
import ee
import os
import shapely
from GEE_functions import GEEUtils 
from GEE_functions import Hazard
from indicator_functions import IndicatorUtils
import datetime as dt

import numpy as np
pd.set_option('display.max_columns', 500)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# ============================================================
# Authenticate and initialize the Earth Engine library
# Define GEE asset path for UNICEF CCRI data
# ============================================================
ee.Authenticate()
ee.Initialize(project="unicef-ccri")
unicef_data_source_path = "projects/unicef-ccri/assets"
utils = GEEUtils(ee)
indicator_utils = IndicatorUtils()

In [ ]:
OUTPUT_FOLDER = f"output"
prod_date = dt.date.today().isoformat()

In [ ]:
PROCESS_SPECIFIC_COUNTRIES = None

In [ ]:
countries_to_process = utils.get_countries_to_process(PROCESS_SPECIFIC_COUNTRIES)
print(f"Countries to process ({len(countries_to_process)}): {countries_to_process}")


In [ ]:
PROCESS_SPECIFIC_HAZARDS = None # Set to a list of specific hazards to process, or None to process all hazards. Example: ['earthquake_pga_0p09', 'fire_frequency']

In [ ]:
# ============================================================
# Load hazard metadata
# ============================================================
hazard_info = pd.read_json("hazard_info.json")
hazard_list = [
    Hazard.from_dict(row, asset_prefix=unicef_data_source_path)
    for row in hazard_info.to_dict(orient="records")
]

for index, hazard in enumerate(hazard_list):
    print(f"{index}. {hazard.name}, {hazard.code}, {hazard.asset}, {hazard.threshold}")

In [ ]:
# Export excel files per country and hazard
for hazard in hazard_list:
    if PROCESS_SPECIFIC_HAZARDS and hazard.code not in PROCESS_SPECIFIC_HAZARDS:
        continue
    try:
        hazard_code = hazard.code
        hazard_name:str = hazard.name
        global_file_path = f"{OUTPUT_FOLDER}/global/CSV/{hazard_code}_{prod_date}.csv"
        df_global = pd.read_csv(global_file_path, index_col=0)

        for country_ucode in countries_to_process:
            try:
                ucode_col = "adm0_ucode"
                df_country:pd.DataFrame = df_global.loc[df_global[ucode_col] == country_ucode]
                df_country.reset_index(drop=True, inplace=True)
                utils.save_country_results_to_excel(df_country, hazard_code, country_ucode, OUTPUT_FOLDER)
            except Exception as e:
                print(f"Error {e} with country {country_ucode}")
    except Exception as e:
        print(f"Error reading file for hazard {hazard_name}: {e}")


In [ ]:
# Export global Excel files from global CSVs
global_excel_folder = f"{OUTPUT_FOLDER}/global/excel"
os.makedirs(global_excel_folder, exist_ok=True)

for hazard in hazard_list:
    if PROCESS_SPECIFIC_HAZARDS and hazard.code not in PROCESS_SPECIFIC_HAZARDS:
        continue
    hazard_code = hazard.code
    hazard_name = hazard.name
    csv_path = f"{OUTPUT_FOLDER}/global/CSV/{hazard_code}_{prod_date}.csv"
    try:
        df_global = pd.read_csv(csv_path, index_col=0)
        xlsx_path = f"{global_excel_folder}/{hazard_code}_{prod_date}.xlsx"
        df_global.to_excel(xlsx_path, index=False)
        print(f"Exported: {xlsx_path} ({len(df_global)} rows)")
    except Exception as e:
        print(f"Error exporting {hazard_name}: {e}")

print("Done – global Excel files exported.")

In [ ]:
def polygons_only(geom:shapely.Geometry):
    # If it is not a collection, just return it as-is
    if geom.geom_type != "GeometryCollection":
        return geom

    # Filter only Polygon / MultiPolygon parts
    polys = []
    for g in geom.geoms:  # GeometryCollection exposes its members via .geoms [web:1]
        if g.geom_type == "Polygon":
            polys.append(g)
        elif g.geom_type == "MultiPolygon":
            polys.extend(list(g.geoms))  # flatten nested MultiPolygons [web:10]

    if not polys:
        return None  # or geom, or an empty GeometryCollection, depending on your needs

    if len(polys) == 1:
        return polys[0]  # single Polygon
    else:
        return shapely.geometry.MultiPolygon(polys)  # build a MultiPolygon from a list of Polygons 

In [ ]:
# Export as Geopackage (GPKG)
adm0_ucode_col = "adm0_ucode"

for country_ucode in countries_to_process:
    # for each country a new adm2 dataset is built
    all_adm2_country = None

    # Save as gpkg files
    for hazard in hazard_list:
        if PROCESS_SPECIFIC_HAZARDS and hazard.code not in PROCESS_SPECIFIC_HAZARDS:
            continue
        hazard_code = hazard.code
        hazard_name:str = hazard.name
        print(f"Starting Geopackage save for: {hazard_name} - {country_ucode}")
        # import the global one
        df_global = pd.read_csv(f"{OUTPUT_FOLDER}/global/CSV/{hazard_code}_{prod_date}.csv", index_col=0)
        # get the country-specific data
        df_country:pd.DataFrame = df_global.loc[df_global[adm0_ucode_col] == country_ucode]
        df_country.reset_index(drop=True, inplace=True)

        # First time, fetch the geometries from GEE
        if all_adm2_country is None:
            print(f"extracting adm2 features for {country_ucode}")
            # build geometries for each adm2 in the country
            adm2_gee_asset = ee.FeatureCollection(f'{unicef_data_source_path}/{utils.ADMIN2_ASSET}')
            adm1_list = list(df_country["adm1_ucode"].unique())

            # get all adm2 geometries by fetching by adm1
            all_adm2_list = []
            for adm1 in adm1_list:
                features = adm2_gee_asset.filter(ee.Filter.eq("adm1_ucode", adm1)).getInfo()["features"]
                partial_adm2 = []
                for f in features:
                    geometry = shapely.geometry.shape(f["geometry"])
                    geometry = polygons_only(geometry)

                    formatted_feature = {"geometry": geometry}
                    formatted_feature.update(f["properties"])
                    partial_adm2.append(formatted_feature)
                
                all_adm2_list += partial_adm2

            all_adm2_country = pd.DataFrame(all_adm2_list)

        cols = list(df_country.columns) + ["geometry"]
        df_country = pd.merge(df_country, all_adm2_country, "right")
        df_country = df_country[cols]
        geodf_country = geopd.GeoDataFrame(df_country, geometry="geometry")
        geodf_country.set_crs(epsg=4326, inplace=True)

            
        country_folder_path = f"{OUTPUT_FOLDER}/results/{country_ucode}/geopackage"
        # create folder if it dows not exist
        os.makedirs(country_folder_path, exist_ok=True)
        # save as Geopackage
        out_gpkg = f"{country_folder_path}/{country_ucode}_{hazard_code}_{prod_date}.gpkg"
        geodf_country.to_file(out_gpkg, index=False, driver="GPKG", layer=hazard_code)
        print(f"Exported {hazard_code} results for country {country_ucode}: {out_gpkg}")
        print()



In [ ]:
# Export global geopackage per hazard by merging per-country geopackages
# Needs the result from the previous cell to have run successfully for all countries to ensure all geometries are available locally
global_gpkg_folder = f"{OUTPUT_FOLDER}/global/geopackage"
os.makedirs(global_gpkg_folder, exist_ok=True)

for hazard in hazard_list:
    if PROCESS_SPECIFIC_HAZARDS and hazard.code not in PROCESS_SPECIFIC_HAZARDS:
        continue
    hazard_code = hazard.code
    hazard_name = hazard.name
    print(f"Merging global geopackage for: {hazard_name} ({hazard_code})")

    frames = []
    for country_ucode in countries_to_process:
        gpkg_path = f"{OUTPUT_FOLDER}/results/{country_ucode}/geopackage/{country_ucode}_{hazard_code}_{prod_date}.gpkg"
        if os.path.exists(gpkg_path):
            gdf = geopd.read_file(gpkg_path, layer=hazard_code)
            frames.append(gdf)
        else:
            print(f"  Warning: missing {gpkg_path}")

    if frames:
        gdf_global = pd.concat(frames, ignore_index=True)
        gdf_global = geopd.GeoDataFrame(gdf_global, geometry="geometry")
        gdf_global.set_crs(epsg=4326, inplace=True)

        out_path = f"{global_gpkg_folder}/{hazard_code}_{prod_date}.gpkg"
        gdf_global.to_file(out_path, index=False, driver="GPKG", layer=hazard_code)
        print(f"  Exported: {out_path} ({len(gdf_global)} rows)")
    else:
        print(f"  No country files found for {hazard_code}, skipping.")

print("Done – global geopackages exported.")